# Huawei Technology Lab 4
## Huawei Cloud ModelArts Workflow for a MindSpore Healthcare Model

**Purpose:** Learn the reproducible cloud workflow used to prepare,
execute, evaluate, and save a MindSpore teaching model in Huawei Cloud
ModelArts.

**Important:** This notebook is a guided workflow. Actual execution in
ModelArts requires a Huawei Cloud account, ModelArts availability in the
chosen region, and suitable compute/billing or teaching credits.

**Educational prototype — not for clinical diagnosis or treatment.**

## Learning objectives

By the end of this lab, the learner should be able to:

1. describe the role of ModelArts in a cloud AI workflow;
2. open or create a JupyterLab notebook environment;
3. verify the installed MindSpore version and device target;
4. prepare a small healthcare teaching dataset;
5. train or load a MindSpore model;
6. save a checkpoint and an evidence report;
7. document the environment so another learner can reproduce the lab.

## Step 1 — In ModelArts

In the Huawei Cloud console:

**ModelArts → DevEnviron / Notebook → create or open a JupyterLab notebook**

Record:
- region;
- notebook image;
- Python version;
- MindSpore version;
- selected device/compute specification;
- date of execution.

Do not upload identifiable patient data.

In [ ]:
# Run this cell inside the selected notebook environment.
import json
import platform
from datetime import datetime

try:
    import mindspore as ms
    mindspore_version = ms.__version__
    device_target = ms.get_context("device_target")
except Exception as exc:
    mindspore_version = f"MindSpore not ready: {exc}"
    device_target = "unknown"

environment_record = {
    "execution_date": datetime.now().isoformat(timespec="seconds"),
    "python_version": platform.python_version(),
    "mindspore_version": mindspore_version,
    "device_target": device_target,
    "clinical_use_allowed": False,
}

print(json.dumps(environment_record, indent=2))

## Step 2 — Use a non-identifiable teaching dataset

The example below creates synthetic numerical healthcare-like data so
the workflow can be practised without transferring patient records.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
X = rng.normal(size=(160, 8)).astype(np.float32)
y = (X[:, 0] + 0.7 * X[:, 1] - 0.4 * X[:, 2] > 0).astype(np.int32)

split = 128
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("Training:", X_train.shape)
print("Test:", X_test.shape)

## Step 3 — Build a small MindSpore model

In [ ]:
import mindspore as ms
from mindspore import nn, ops
import mindspore.dataset as ds

ms.set_seed(42)

train_ds = ds.NumpySlicesDataset(
    (X_train, y_train),
    column_names=["features", "label"],
    shuffle=True,
).batch(16)

class SmallNet(nn.Cell):
    def __init__(self):
        super().__init__()
        self.net = nn.SequentialCell(
            nn.Dense(8, 16),
            nn.ReLU(),
            nn.Dense(16, 2),
        )

    def construct(self, x):
        return self.net(x)

net = SmallNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = nn.Adam(net.trainable_params(), learning_rate=0.001)

def forward_fn(features, labels):
    logits = net(features)
    loss = loss_fn(logits, labels)
    return loss, logits

grad_fn = ms.value_and_grad(
    forward_fn, None, optimizer.parameters, has_aux=True
)

def train_step(features, labels):
    (loss, logits), grads = grad_fn(features, labels)
    optimizer(grads)
    return loss

## Step 4 — Train briefly and save the checkpoint

In [ ]:
from pathlib import Path

for epoch in range(5):
    losses = []
    for features, labels in train_ds.create_tuple_iterator():
        losses.append(float(train_step(features, labels).asnumpy()))
    print(f"Epoch {epoch + 1}: loss={np.mean(losses):.4f}")

output_dir = Path("modelarts_lab_outputs")
output_dir.mkdir(exist_ok=True)

ms.save_checkpoint(net, str(output_dir / "modelarts_healthcare_demo.ckpt"))

(output_dir / "environment.json").write_text(
    json.dumps(environment_record, indent=2),
    encoding="utf-8",
)

print("Saved:", output_dir.resolve())

## Competition evidence checklist

If this lab is actually run in ModelArts, capture:

- screenshot of ModelArts notebook environment;
- MindSpore version;
- device target;
- successful training output;
- saved checkpoint;
- environment JSON;
- date/region/image used.

Until those items exist, describe this as a **ModelArts-ready guided
workflow**, not as completed student execution.